# Quantization-Aware Training (QAT) Sweep (QConv2D Architecture)

Kernel: **qkeras-tf215** (TF 2.15 + Keras 2 + QKeras)

Sweeps all 16 (w_bits, a_bits) from {2,4,8,32}^2 x CNN/RCNN x 12 (d,p) configs.

Run order: Cell 1 -> 2 -> 3 with VALIDATE_ONLY=True first, then False for full sweep.

In [2]:
import os
import csv
import time
import tempfile
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from qkeras import QDense, QConv2D, QActivation, quantized_bits
from datetime import datetime

print(f'TF: {tf.__version__}')
import qkeras as _qk
print(f'QKeras: {_qk.__version__}')

TF: 2.21.0
QKeras: 0.9.0


In [3]:
CONFIGS = [
    (3, 0.001), (3, 0.005), (3, 0.010), (3, 0.050),
    (5, 0.001), (5, 0.005), (5, 0.010), (5, 0.050),
    (7, 0.001), (7, 0.005), (7, 0.010), (7, 0.050),
]
ROUNDS        = 2
DATA_DIR      = './datasets'
OUTPUT_DIR    = './results'
SEED          = 42
BATCH_SIZE    = 256
LEARNING_RATE = 1e-3
PATIENCE      = 10
MAX_EPOCHS    = 100

BIT_WIDTHS    = [2, 4, 8, 32]
QUANT_CONFIGS = [(w, a) for w in BIT_WIDTHS for a in BIT_WIDTHS]

os.makedirs(OUTPUT_DIR, exist_ok=True)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'{len(QUANT_CONFIGS)} quant configs x {len(CONFIGS)} datasets x 2 arches = {len(QUANT_CONFIGS)*len(CONFIGS)*2} total runs')

16 quant configs x 12 datasets x 2 arches = 384 total runs


In [7]:
def get_2d_shape(n):
    side = int(np.sqrt(n))
    if side * side == n:
        return side, side
    for h in range(side, 0, -1):
        if n % h == 0:
            return h, n // h
    return 1, n


def build_quantized_cnn(input_shape, output_shape, w_bits, a_bits):
    H, W = get_2d_shape(input_shape[0])
    w_q = quantized_bits(w_bits, 1)
    a_q = f'quantized_relu({a_bits})'

    if w_bits == 32:
        model = keras.Sequential([
            keras.layers.Input(shape=input_shape),
            keras.layers.Reshape((H, W, 1)),
            keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
            keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
            keras.layers.Flatten(),
            keras.layers.Dense(64, activation='relu'),
            keras.layers.Dense(output_shape[0], activation='sigmoid'),
        ])
    else:
        model = keras.Sequential([
            keras.layers.Input(shape=input_shape),
            keras.layers.Reshape((H, W, 1)),
            QConv2D(32, 3, padding='same',
                    kernel_quantizer=w_q, bias_quantizer=w_q, activation=None),
            QActivation(a_q),
            QConv2D(32, 3, padding='same',
                    kernel_quantizer=w_q, bias_quantizer=w_q, activation=None),
            QActivation(a_q),
            keras.layers.Flatten(),
            QDense(64, kernel_quantizer=w_q, bias_quantizer=w_q),
            QActivation(a_q),
            keras.layers.Dense(output_shape[0], activation='sigmoid'),
        ])
    return model


def build_quantized_rcnn(input_shape, output_shape, w_bits, a_bits):
    H, W = get_2d_shape(input_shape[0])
    w_q = quantized_bits(w_bits, 1)
    a_q = f'quantized_relu({a_bits})'

    if w_bits == 32:
        model = keras.Sequential([
            keras.layers.Input(shape=input_shape),
            keras.layers.Reshape((H, W, 1)),
            keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
            keras.layers.Flatten(),
            keras.layers.RepeatVector(1),
            keras.layers.LSTM(32, activation='relu'),
            keras.layers.Dense(64, activation='relu'),
            keras.layers.Dense(output_shape[0], activation='sigmoid'),
        ])
    else:
        model = keras.Sequential([
            keras.layers.Input(shape=input_shape),
            keras.layers.Reshape((H, W, 1)),
            QConv2D(32, 3, padding='same',
                    kernel_quantizer=w_q, bias_quantizer=w_q, activation=None),
            QActivation(a_q),
            keras.layers.Flatten(),
            keras.layers.RepeatVector(1),
            keras.layers.LSTM(32, activation='tanh'),
            QDense(64, kernel_quantizer=w_q, bias_quantizer=w_q),
            QActivation(a_q),
            keras.layers.Dense(output_shape[0], activation='sigmoid'),
        ])
    return model


def measure_model_size_kb(model):
    with tempfile.NamedTemporaryFile(suffix='.h5', delete=False) as f:
        tmp = f.name
    model.save(tmp)
    kb = os.path.getsize(tmp) / 1024
    os.remove(tmp)
    return kb


def measure_inference_latency_ms(model, X_sample, n_repeats=10):
    batch = X_sample[:1000]
    model.predict(batch, verbose=0)  # warm-up
    times = []
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        model.predict(batch, verbose=0)
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(times))


# Sanity-check reshape for all distances
for d in [3, 5, 7]:
    n = (d**2 - 1) * ROUNDS
    H, W = get_2d_shape(n)
    assert H * W == n, f'd={d}: {H}x{W}={H*W} != {n}'
    print(f'd={d}: {n} features -> ({H},{W},1)')
print('Builders ready.')

d=3: 16 features -> (4,4,1)
d=5: 48 features -> (6,8,1)
d=7: 96 features -> (8,12,1)
Builders ready.


In [8]:
# Set True to validate pipeline on d=5,p=0.01 only (32 runs).
# Set False for the full 384-run sweep.
VALIDATE_ONLY = True

sweep_configs = [(5, 0.010)] if VALIDATE_ONLY else CONFIGS

csv_path   = f'{OUTPUT_DIR}/results_quantization_sweep.csv'
fieldnames = ['architecture', 'w_bits', 'a_bits', 'd', 'p',
              'p_L', 'model_size_kb', 'inference_latency_ms']

# Write header (overwrites any previous file)
with open(csv_path, 'w', newline='') as f:
    csv.DictWriter(f, fieldnames=fieldnames).writeheader()

results    = []
total_runs = len(sweep_configs) * len(QUANT_CONFIGS) * 2
run_idx    = 0
t_start    = datetime.now()

for d, p in sweep_configs:
    data     = np.load(f'{DATA_DIR}/data_d{d}_p{p:.3f}_r{ROUNDS}.npz')
    det_evts = data['det_evts'].astype(np.float32)
    flips    = data['flips'].astype(np.float32)

    n_train, n_val = 800_000, 100_000
    X_train, y_train = det_evts[:n_train],              flips[:n_train]
    X_val,   y_val   = det_evts[n_train:n_train+n_val], flips[n_train:n_train+n_val]
    X_test,  y_test  = det_evts[n_train+n_val:],        flips[n_train+n_val:]
    input_shape  = X_train.shape[1:]
    output_shape = y_train.shape[1:]

    print(f'\n{"="*65}')
    print(f'd={d} p={p:.3f} | train={X_train.shape} val={X_val.shape} test={X_test.shape}')
    print(f'{"="*65}')

    for w_bits, a_bits in QUANT_CONFIGS:
        for arch_name, build_fn in [('CNN',  build_quantized_cnn),
                                    ('RCNN', build_quantized_rcnn)]:
            run_idx += 1
            print(f'\n  [{run_idx}/{total_runs}] {arch_name} w={w_bits} a={a_bits} | d={d} p={p:.3f}')
            try:
                model = build_fn(input_shape, output_shape, w_bits, a_bits)
                model.compile(
                    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                    loss='binary_crossentropy',
                    metrics=['accuracy']
                )
                model.fit(
                    X_train, y_train,
                    batch_size=BATCH_SIZE,
                    validation_data=(X_val, y_val),
                    epochs=MAX_EPOCHS,
                    callbacks=[keras.callbacks.EarlyStopping(
                        patience=PATIENCE, restore_best_weights=True)],
                    verbose=1
                )

                pred = (model.predict(X_test, verbose=0) > 0.5).astype(float)
                p_L  = float((pred != y_test).sum()) / len(y_test)
                kb   = measure_model_size_kb(model)
                lat  = measure_inference_latency_ms(model, X_test)

                print(f'  -> p_L={p_L:.6f}  size={kb:.1f}KB  latency={lat:.1f}ms')

                row = {'architecture': arch_name, 'w_bits': w_bits, 'a_bits': a_bits,
                       'd': d, 'p': p, 'p_L': round(p_L, 8),
                       'model_size_kb': round(kb, 2),
                       'inference_latency_ms': round(lat, 2)}
                results.append(row)
                with open(csv_path, 'a', newline='') as f:
                    csv.DictWriter(f, fieldnames=fieldnames).writerow(row)

            except Exception as e:
                print(f'  ERROR: {e}')
            finally:
                try:
                    del model
                except NameError:
                    pass
                keras.backend.clear_session()

print(f'\nDone: {len(results)}/{total_runs} runs succeeded.')
print(f'Elapsed: {datetime.now() - t_start}')
print(f'CSV: {csv_path}')


d=5 p=0.010 | train=(800000, 48) val=(100000, 48) test=(100000, 48)

  [1/32] CNN w=2 a=2 | d=5 p=0.010
  ERROR: The added layer must be an instance of class Layer. Received: layer=<QConv2D name=q_conv2d_36, built=False> of type <class 'qkeras.qconvolutional.QConv2D'>.

  [2/32] RCNN w=2 a=2 | d=5 p=0.010
  ERROR: The added layer must be an instance of class Layer. Received: layer=<QConv2D name=q_conv2d_38, built=False> of type <class 'qkeras.qconvolutional.QConv2D'>.

  [3/32] CNN w=2 a=4 | d=5 p=0.010
  ERROR: The added layer must be an instance of class Layer. Received: layer=<QConv2D name=q_conv2d_39, built=False> of type <class 'qkeras.qconvolutional.QConv2D'>.

  [4/32] RCNN w=2 a=4 | d=5 p=0.010
  ERROR: The added layer must be an instance of class Layer. Received: layer=<QConv2D name=q_conv2d_41, built=False> of type <class 'qkeras.qconvolutional.QConv2D'>.

  [5/32] CNN w=2 a=8 | d=5 p=0.010
  ERROR: The added layer must be an instance of class Layer. Received: layer=<QConv2D

Epoch 1/100
3125/3125 [==============================] - 20s 6ms/step - loss: 0.2637 - accuracy: 0.8883 - val_loss: 0.2056 - val_accuracy: 0.9134
Epoch 2/100
3125/3125 [==============================] - 20s 6ms/step - loss: 0.1844 - accuracy: 0.9216 - val_loss: 0.1728 - val_accuracy: 0.9268
Epoch 3/100
3125/3125 [==============================] - 20s 6ms/step - loss: 0.1599 - accuracy: 0.9326 - val_loss: 0.1548 - val_accuracy: 0.9351
Epoch 4/100
3125/3125 [==============================] - 20s 6ms/step - loss: 0.1487 - accuracy: 0.9372 - val_loss: 0.1486 - val_accuracy: 0.9370
Epoch 5/100
3125/3125 [==============================] - 20s 6ms/step - loss: 0.1422 - accuracy: 0.9400 - val_loss: 0.1456 - val_accuracy: 0.9394
Epoch 6/100
3125/3125 [==============================] - 20s 6ms/step - loss: 0.1373 - accuracy: 0.9427 - val_loss: 0.1421 - val_accuracy: 0.9417
Epoch 7/100
3125/3125 [==============================] - 20s 6ms/step - loss: 0.1330 - accuracy: 0.9447 - val_loss: 0.1417 -

/opt/anaconda3/envs/qkeras-decoder/lib/python3.10/site-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


  -> p_L=0.050540  size=1307.1KB  latency=38.5ms

  [26/32] RCNN w=32 a=2 | d=5 p=0.010
Epoch 1/100
3125/3125 [==============================] - 13s 4ms/step - loss: 0.2248 - accuracy: 0.8955 - val_loss: 0.1741 - val_accuracy: 0.9227
Epoch 2/100
3125/3125 [==============================] - 12s 4ms/step - loss: 0.1591 - accuracy: 0.9293 - val_loss: 0.1491 - val_accuracy: 0.9345
Epoch 3/100
3125/3125 [==============================] - 13s 4ms/step - loss: 0.1432 - accuracy: 0.9377 - val_loss: 0.1367 - val_accuracy: 0.9406
Epoch 4/100
3125/3125 [==============================] - 13s 4ms/step - loss: 0.1303 - accuracy: 0.9445 - val_loss: 0.1257 - val_accuracy: 0.9465
Epoch 5/100
3125/3125 [==============================] - 12s 4ms/step - loss: 0.1219 - accuracy: 0.9487 - val_loss: 0.1203 - val_accuracy: 0.9496
Epoch 6/100
3125/3125 [==============================] - 12s 4ms/step - loss: 0.1164 - accuracy: 0.9509 - val_loss: 0.1171 - val_accuracy: 0.9506
Epoch 7/100
3125/3125 [=============

  -> p_L=0.035220  size=2429.7KB  latency=32.1ms

  [27/32] CNN w=32 a=4 | d=5 p=0.010
Epoch 1/100
3125/3125 [==============================] - 24s 7ms/step - loss: 0.2641 - accuracy: 0.8847 - val_loss: 0.2099 - val_accuracy: 0.9114
Epoch 2/100
3125/3125 [==============================] - 23s 7ms/step - loss: 0.1867 - accuracy: 0.9197 - val_loss: 0.1748 - val_accuracy: 0.9256
Epoch 3/100
3125/3125 [==============================] - 25s 8ms/step - loss: 0.1621 - accuracy: 0.9314 - val_loss: 0.1568 - val_accuracy: 0.9349
Epoch 4/100
3125/3125 [==============================] - 24s 8ms/step - loss: 0.1487 - accuracy: 0.9389 - val_loss: 0.1514 - val_accuracy: 0.9371
Epoch 5/100
3125/3125 [==============================] - 26s 8ms/step - loss: 0.1403 - accuracy: 0.9426 - val_loss: 0.1442 - val_accuracy: 0.9406
Epoch 6/100
3125/3125 [==============================] - 28s 9ms/step - loss: 0.1348 - accuracy: 0.9453 - val_loss: 0.1405 - val_accuracy: 0.9434
Epoch 7/100
3125/3125 [==============

  -> p_L=0.053690  size=1307.1KB  latency=37.4ms

  [28/32] RCNN w=32 a=4 | d=5 p=0.010
Epoch 1/100
3125/3125 [==============================] - 17s 5ms/step - loss: 0.2127 - accuracy: 0.9060 - val_loss: 0.1597 - val_accuracy: 0.9309
Epoch 2/100
3125/3125 [==============================] - 19s 6ms/step - loss: 0.1394 - accuracy: 0.9415 - val_loss: 0.1286 - val_accuracy: 0.9457
Epoch 3/100
3125/3125 [==============================] - 20s 6ms/step - loss: 0.1161 - accuracy: 0.9521 - val_loss: 0.1033 - val_accuracy: 0.9582
Epoch 4/100
3125/3125 [==============================] - 20s 6ms/step - loss: 0.0967 - accuracy: 0.9618 - val_loss: 0.0916 - val_accuracy: 0.9649
Epoch 5/100
3125/3125 [==============================] - 19s 6ms/step - loss: 0.0881 - accuracy: 0.9662 - val_loss: 0.0883 - val_accuracy: 0.9646
Epoch 6/100
3125/3125 [==============================] - 19s 6ms/step - loss: 0.0837 - accuracy: 0.9679 - val_loss: 0.0845 - val_accuracy: 0.9681
Epoch 7/100
3125/3125 [=============

  -> p_L=0.030070  size=2429.7KB  latency=45.0ms

  [29/32] CNN w=32 a=8 | d=5 p=0.010
Epoch 1/100
3125/3125 [==============================] - 31s 10ms/step - loss: 0.2428 - accuracy: 0.8921 - val_loss: 0.1969 - val_accuracy: 0.9149
Epoch 2/100
3125/3125 [==============================] - 31s 10ms/step - loss: 0.1769 - accuracy: 0.9223 - val_loss: 0.1662 - val_accuracy: 0.9269
Epoch 3/100
3125/3125 [==============================] - 31s 10ms/step - loss: 0.1591 - accuracy: 0.9319 - val_loss: 0.1550 - val_accuracy: 0.9334
Epoch 4/100
3125/3125 [==============================] - 33s 11ms/step - loss: 0.1506 - accuracy: 0.9365 - val_loss: 0.1523 - val_accuracy: 0.9367
Epoch 5/100
3125/3125 [==============================] - 31s 10ms/step - loss: 0.1446 - accuracy: 0.9394 - val_loss: 0.1483 - val_accuracy: 0.9380
Epoch 6/100
3125/3125 [==============================] - 30s 10ms/step - loss: 0.1397 - accuracy: 0.9419 - val_loss: 0.1433 - val_accuracy: 0.9409
Epoch 7/100
3125/3125 [========

  -> p_L=0.051970  size=1307.1KB  latency=57.2ms

  [30/32] RCNN w=32 a=8 | d=5 p=0.010
Epoch 1/100
3125/3125 [==============================] - 21s 6ms/step - loss: 0.1927 - accuracy: 0.9160 - val_loss: 0.1330 - val_accuracy: 0.9446
Epoch 2/100
3125/3125 [==============================] - 20s 6ms/step - loss: 0.1169 - accuracy: 0.9530 - val_loss: 0.1101 - val_accuracy: 0.9570
Epoch 3/100
3125/3125 [==============================] - 20s 6ms/step - loss: 0.0977 - accuracy: 0.9614 - val_loss: 0.0927 - val_accuracy: 0.9632
Epoch 4/100
3125/3125 [==============================] - 21s 7ms/step - loss: 0.0904 - accuracy: 0.9646 - val_loss: 0.0919 - val_accuracy: 0.9626
Epoch 5/100
3125/3125 [==============================] - 20s 6ms/step - loss: 0.0860 - accuracy: 0.9666 - val_loss: 0.0872 - val_accuracy: 0.9668
Epoch 6/100
3125/3125 [==============================] - 21s 7ms/step - loss: 0.0828 - accuracy: 0.9682 - val_loss: 0.0831 - val_accuracy: 0.9677
Epoch 7/100
3125/3125 [=============

  -> p_L=0.029380  size=2429.7KB  latency=44.4ms

  [31/32] CNN w=32 a=32 | d=5 p=0.010
Epoch 1/100
3125/3125 [==============================] - 32s 10ms/step - loss: 0.2531 - accuracy: 0.8882 - val_loss: 0.1915 - val_accuracy: 0.9171
Epoch 2/100
3125/3125 [==============================] - 31s 10ms/step - loss: 0.1755 - accuracy: 0.9257 - val_loss: 0.1694 - val_accuracy: 0.9297
Epoch 3/100
3125/3125 [==============================] - 31s 10ms/step - loss: 0.1616 - accuracy: 0.9327 - val_loss: 0.1582 - val_accuracy: 0.9345
Epoch 4/100
3125/3125 [==============================] - 32s 10ms/step - loss: 0.1535 - accuracy: 0.9366 - val_loss: 0.1546 - val_accuracy: 0.9373
Epoch 5/100
3125/3125 [==============================] - 30s 10ms/step - loss: 0.1466 - accuracy: 0.9400 - val_loss: 0.1508 - val_accuracy: 0.9376
Epoch 6/100
3125/3125 [==============================] - 31s 10ms/step - loss: 0.1422 - accuracy: 0.9419 - val_loss: 0.1475 - val_accuracy: 0.9416
Epoch 7/100
3125/3125 [=======

  -> p_L=0.054650  size=1307.1KB  latency=66.7ms

  [32/32] RCNN w=32 a=32 | d=5 p=0.010
Epoch 1/100
3125/3125 [==============================] - 21s 6ms/step - loss: 0.1936 - accuracy: 0.9130 - val_loss: 0.1335 - val_accuracy: 0.9420
Epoch 2/100
3125/3125 [==============================] - 20s 7ms/step - loss: 0.1214 - accuracy: 0.9482 - val_loss: 0.1198 - val_accuracy: 0.9496
Epoch 3/100
3125/3125 [==============================] - 21s 7ms/step - loss: 0.1086 - accuracy: 0.9560 - val_loss: 0.1021 - val_accuracy: 0.9589
Epoch 4/100
3125/3125 [==============================] - 21s 7ms/step - loss: 0.0997 - accuracy: 0.9606 - val_loss: 0.0967 - val_accuracy: 0.9621
Epoch 5/100
3125/3125 [==============================] - 21s 7ms/step - loss: 0.0937 - accuracy: 0.9634 - val_loss: 0.0919 - val_accuracy: 0.9642
Epoch 6/100
3125/3125 [==============================] - 20s 7ms/step - loss: 0.0895 - accuracy: 0.9650 - val_loss: 0.0914 - val_accuracy: 0.9649
Epoch 7/100
3125/3125 [============

In [9]:
import pandas as pd

df = pd.read_csv(csv_path)
print(f'Total rows: {len(df)}')

for (d_val, p_val), grp in df.groupby(['d', 'p']):
    print(f'\nd={d_val}, p={p_val:.3f}:')
    print(grp[['architecture', 'w_bits', 'a_bits', 'p_L', 'model_size_kb', 'inference_latency_ms']]
          .sort_values(['architecture', 'w_bits', 'a_bits'])
          .to_string(index=False))

Total rows: 8

d=5, p=0.010:
architecture  w_bits  a_bits     p_L  model_size_kb  inference_latency_ms
         CNN      32       2 0.05054        1307.08                 38.46
         CNN      32       4 0.05369        1307.08                 37.41
         CNN      32       8 0.05197        1307.08                 57.22
         CNN      32      32 0.05465        1307.08                 66.69
        RCNN      32       2 0.03522        2429.73                 32.05
        RCNN      32       4 0.03007        2429.73                 44.99
        RCNN      32       8 0.02938        2429.73                 44.40
        RCNN      32      32 0.03064        2429.73                 36.91
